In [1]:
# Cell 1: Configuration and Setup
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime
 
# Configuration
BRONZE_PATH = "Files/bronze"
SILVER_PATH = "Files/silver"
CHECKPOINT_PATH = "Files/checkpoints"
 
print("✅ Configuration loaded")
print(f"📅 Pipeline run time: {datetime.now()}")


StatementMeta(, 7272f26f-6577-4cd8-9d87-27c24ce2ba4a, 3, Finished, Available, Finished, False)

✅ Configuration loaded
📅 Pipeline run time: 2026-05-08 05:02:36.191631


In [ ]:
# Cell 2: Read Bronze Layer Data
 
# Read Sales
bronze_sales_df = (
    spark.read
    .format("parquet")
    .load(f"{BRONZE_PATH}/sales/*")
)
 
# Read Products
bronze_products_df = (
    spark.read
    .format("parquet")
    .load(f"{BRONZE_PATH}/products/*")
)
 
# Read Customers
bronze_customers_df = (
    spark.read
    .format("parquet")
    .load(f"{BRONZE_PATH}/customers/*")
)
 
print(f"📊 Sales records: {bronze_sales_df.count():,}")
print(f"📦 Products: {bronze_products_df.count():,}")
print(f"👥 Customers: {bronze_customers_df.count():,}")
 
# Display sample
display (bronze_sales_df)

In [ ]:
# Cell 3: Data Quality Validation Functions

def validate_data_quality(df, dataset_name):
    """
    Comprehensive data quality checks
    """
    total_records = df.count()
    
    print(f"\n{'='*60}")
    print(f"📋 Data Quality Report: {dataset_name}")
    print(f"{'='*60}")
    
    # 1. Completeness Check
    print("\n1️⃣ COMPLETENESS CHECK:")
    null_counts = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    
    for col, null_count in null_counts.items():
        null_pct = (null_count / total_records) * 100
        status = "✅" if null_pct < 5 else "⚠️" if null_pct < 10 else "❌"
        print(f" {status} {col}: {null_count} nulls ({null_pct:.2f}%)")
    
    # 2. Duplicates Check
    print("\n2️⃣ DUPLICATES CHECK:")
    duplicate_count = total_records - df.dropDuplicates().count()
    dup_pct = (duplicate_count / total_records) * 100
    status = "✅" if dup_pct == 0 else "⚠️"
    print(f" {status} Duplicates: {duplicate_count} ({dup_pct:.2f}%)")
    
    # 3. Overall Quality Score
    print(f"\n{'='*60}")
    quality_issues = sum(1 for v in null_counts.values() if v > 0) + (1 if duplicate_count > 0 else 0)
    quality_score = max(0, 100 - (quality_issues * 10))
    print(f"📊 Overall Quality Score: {quality_score}/100")
    print(f"{'='*60}\n")
    
    return quality_score

# Run quality checks
validate_data_quality(bronze_sales_df, "Sales")
validate_data_quality(bronze_products_df, "Products")
validate_data_quality(bronze_customers_df, "Customers")

In [ ]:
# Cell 4: Transform Sales to Silver Layer

silver_sales_df = (
    bronze_sales_df
    
    # Data Cleansing
    .filter(F.col("Order ID").isNotNull()) # Remove nulls in critical field
    .dropDuplicates(["Order ID", "Product ID"]) # Remove duplicates
    
    # Standardization
    .withColumn("order_date", F.to_date(F.col("Order Date"), "M/d/yyyy"))
    .withColumn("ship_date", F.to_date(F.col("Ship Date"), "M/d/yyyy"))
    .withColumn("sales", F.col("Sales").cast("decimal(10,2)"))
    .withColumn("quantity", F.col("Quantity").cast("int"))
    .withColumn("discount", F.col("Discount").cast("decimal(4,2)"))
    .withColumn("profit", F.col("Profit").cast("decimal(10,2)"))
    
    # Derived Columns
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("order_quarter", F.quarter("order_date"))
    .withColumn("order_day_of_week", F.dayofweek("order_date"))
    
    # Calculate metrics
    .withColumn("profit_margin", 
                F.when(F.col("sales") > 0, 
                      (F.col("profit") / F.col("sales")) * 100)
                .otherwise(0))
    
    .withColumn("discount_amount", F.col("sales") * F.col("discount"))
    .withColumn("revenue_after_discount", F.col("sales") - F.col("discount_amount"))
    
    # Audit columns
    .withColumn("processed_timestamp", F.current_timestamp())
    .withColumn("data_quality_flag", F.lit("PASSED"))
    .withColumn("source_system", F.lit("BRONZE_SALES"))
    
    # Column selection and renaming
    .select(
        F.col("Row ID").alias("row_id"),
        F.col("Order ID").alias("order_id"),
        F.col("order_date"),
        F.col("ship_date"),
        F.col("Ship Mode").alias("ship_mode"),
        F.col("Customer ID").alias("customer_id"),
        F.col("Customer Name").alias("customer_name"),
        F.col("Segment").alias("segment"),
        F.col("Country").alias("country"),
        F.col("City").alias("city"),
        F.col("State").alias("state"),
        F.col("Postal Code").alias("postal_code"),
        F.col("Region").alias("region"),
        F.col("Product ID").alias("product_id"),
        F.col("Category").alias("category"),
        F.col("Sub-Category").alias("subcategory"),
        F.col("Product Name").alias("product_name"),
        "sales",
        "quantity",
        "discount",
        "profit",
        "profit_margin",
        "discount_amount",
        "revenue_after_discount",
        "order_year",
        "order_month",
        "order_quarter",
        "order_day_of_week",
        "processed_timestamp",
        "data_quality_flag",
        "source_system"
    )
)

print(f"✅ Silver Sales Transformed: {silver_sales_df.count():,} records")
display(silver_sales_df)

In [ ]:
#display(bronze_products_df)
#display(bronze_sales_df)

In [ ]:
# Cell 5: Products Dimension from Sales Data 

# Extract unique products from sales data
silver_products_df = (
    bronze_sales_df
    
    # Select product-related columns
    .select(
        F.col("Product ID").alias("product_id"),
        F.col("Product Name").alias("product_name"),
        F.col("Category").alias("category"),
        F.col("Sub-Category").alias("subcategory")
    )
    
    # Remove duplicates - get unique products
    .dropDuplicates(["product_id"])
    
    # Data cleansing
    .filter(F.col("product_id").isNotNull())
    
    # Standardization
    .withColumn("product_name_clean", 
                F.trim(F.lower(F.col("product_name"))))
    .withColumn("category_clean", 
                F.trim(F.initcap(F.col("category"))))
    .withColumn("subcategory_clean", 
                F.trim(F.initcap(F.col("subcategory"))))
    
    # Audit columns
    .withColumn("processed_timestamp", F.current_timestamp())
    .withColumn("is_active", F.lit(True))
    .withColumn("source_system", F.lit("SALES_DATA"))
    
    # Final selection
    .select(
        "product_id",
        F.col("product_name").alias("product_name_original"),
        "product_name_clean",
        F.col("category").alias("category_original"),
        "category_clean",
        F.col("subcategory").alias("subcategory_original"),
        "subcategory_clean",
        "processed_timestamp",
        "is_active",
        "source_system"
    )
)

print(f"✅ Silver Products (from Sales): {silver_products_df.count():,} unique products")
display(silver_products_df)

In [ ]:
# Cell 6: Customers Dimension from Sales Data (NOT separate customers.csv)

# Extract unique customers from sales data
silver_customers_df = (
    bronze_sales_df
    
    # Select customer-related columns
    .select(
        F.col("Customer ID").alias("customer_id"),
        F.col("Customer Name").alias("customer_name"),
        F.col("Segment").alias("segment"),
        F.col("Country").alias("country"),
        F.col("State").alias("state"),
        F.col("City").alias("city")
    )
    
    # Remove duplicates - get unique customers
    .dropDuplicates(["customer_id"])
    
    # Data cleansing
    .filter(F.col("customer_id").isNotNull())
    
    # Standardization
    .withColumn("customer_name_clean", 
                F.trim(F.initcap(F.col("customer_name"))))
    
    # Create synthetic registration date (first appearance in data)
    # In real scenario, you'd get this from a customer master table
    .withColumn("registration_date", F.lit("2020-01-01").cast("date"))
    
    # Derived columns
    .withColumn("days_since_registration", 
                F.datediff(F.current_date(), F.col("registration_date")))
    
    .withColumn("customer_tenure_category",
                F.when(F.col("days_since_registration") < 90, "New")
                .when(F.col("days_since_registration") < 365, "Regular")
                .otherwise("Loyal"))
    
    # Audit columns
    .withColumn("processed_timestamp", F.current_timestamp())
    .withColumn("is_active", F.lit(True))
    .withColumn("source_system", F.lit("SALES_DATA"))
    
    # Final selection
    .select(
        "customer_id",
        F.col("customer_name").alias("customer_name_original"),
        "customer_name_clean",
        "segment",
        "country",
        "state", 
        "city",
        "registration_date",
        "days_since_registration",
        "customer_tenure_category",
        "processed_timestamp",
        "is_active",
        "source_system"
    )
)

print(f"✅ Silver Customers (from Sales): {silver_customers_df.count():,} unique customers")
display(silver_customers_df)

In [ ]:
# ========================================
# CELL 7: Write Silver Delta Tables
# ========================================

print("💾 Writing Silver layer to Delta tables...\n")



# Create managed table pointing to Files location - silver sales table 
silver_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_sales")

print("✅ silver_sales table created")

# Create managed table pointing to Files location - silver products
silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_products")

print("✅ silver_products table created")

# Create managed table pointing to Files location - silver customers
silver_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customers")

print("✅ silver_customers table created")

print("\n" + "="*70)
print("🎉 ALL SILVER DELTA TABLES CREATED SUCCESSFULLY!")
print("="*70)
print("\n📊 Available Tables:")
print(" 1. silver_sales")
print(" 2. silver_products")
print(" 3. silver_customers")

In [ ]:
# ========================================
# CELL 8: Validation and Summary
# ========================================

print("="*70)
print("📊 SILVER LAYER VALIDATION SUMMARY")
print("="*70)

# Verify all tables exist and are accessible
tables_info = [
    ("silver_sales", "Fact table - All transactions"),
    ("silver_products", "Dimension table - Product master"),
    ("silver_customers", "Dimension table - Customer master")
]

for table_name, description in tables_info:
    try:
        df = spark.table(table_name)
        count = df.count()
        columns = len(df.columns)
        
        print(f"\n✅ {table_name}")
        print(f" Description: {description}")
        print(f" Records: {count:,}")
        print(f" Columns: {columns}")
        print(f" Location: Files/silver/{table_name.replace('silver_', '')}")
        
    except Exception as e:
        print(f"\n❌ {table_name} - ERROR: {str(e)}")

print(f"\n{'='*70}")
print("🎉 SILVER LAYER PROCESSING COMPLETE!")
print(f"{'='*70}")